### Serialize and Deserialize Binary Tree (Hard)

Serialization is the process of converting a data structure or object into a sequence of bits so that it can be stored in a file or memory buffer, or transmitted across a network connection link to be reconstructed later in the same or another computer environment.

Design an algorithm to serialize and deserialize a binary tree. There is no restriction on how your serialization/deserialization algorithm should work. You just need to ensure that a binary tree can be serialized to a string and this string can be deserialized to the original tree structure.

Clarification: The input/output format is the same as how LeetCode serializes a binary tree. You do not necessarily need to follow this format, so please be creative and come up with different approaches yourself.

In [19]:
listy = [1,2,3,43,5]
result = ','.join(str(num) for num in listy)
print(result)



1,2,3,43,5


In [12]:
listy = [int(num) for num in result.split(', ')]
print(listy)

[1, 2, 3, 43, 5]


In [13]:
list1 = [1,2,3,4,5]
list2 = [6,7,8,9,10]

In [15]:
str1 = "".join(str(d) for d in list1)
str2 = "".join(str(d) for d in list2)
stringyyy = str1 + "+" + str2
print(stringyyy)

12345+678910


In [16]:
class TreeNode(object):
    def __init__(self, x):
        self.val = x
        self.left = None
        self.right = None


In [18]:
from typing import List

### Brute Force Approach

we take a tree
searialize -> traverse using inorder dfs, store node values in inorder list -> traverse using preorder dfs, store node values in preorder list -> combine the lists and give the output as string

deserialize -> split the string into preorder and postorder lists -> using these 2 lists build up a tree

In [ ]:
class Solution:
    def serialize(self, root) -> str | None:
        stringy = ''
        # if root is None
        if root is None:
            return stringy
        
        inorder_list = []
        preorder_list = []

        def in_order_traversal(root: TreeNode, inorder_list: List) -> None:
            
            # base case if root is returning None we end it
            if root is None:
                return None
            # if root has some value in it then 
            # we first visit the left subtree
            in_order_traversal(root.left, inorder_list)
            inorder_list.append(root.val)
            in_order_traversal(root.right,inorder_list)

        def pre_order_traversal(root: TreeNode, preorder_list: List) -> None:
            
            # base case if root is returning None we end it
            if root is None:
                return None
            # if root has some value in it then 
            # we first visit the left subtree
            preorder_list.append(root.val)
            pre_order_traversal(root.left, preorder_list)
            pre_order_traversal(root.right,preorder_list)

        # now our preorder and inorder lists are full
        in_order_traversal(root, inorder_list)
        pre_order_traversal(root, preorder_list)
        # now our preorder and inorder lists are full
        inorder_string = ','.join(str(num) for num in inorder_list)
        preorder_string = ','.join(str(num) for num in preorder_list)

        stringy = inorder_string+"+"+preorder_string
        return stringy

    def deserialize(self, stringy:str) -> TreeNode | None:

        if len(stringy) == 0:
            return None
        
        # 1. Split into two string parts by '+'
        inorder_string, preorder_string = stringy.split('+')

        # 2. Split each part by ',' and convert each item to int
        inorder_list = [int(num) for num in inorder_string.split(',')]
        preorder_list = [int(num) for num in preorder_string.split(',')]

        def buildTree(self, preorder: list[int], inorder: list[int]) -> TreeNode | None:

            # if base case, lists are empty
            if not preorder:
                return None

            # otherwise the first element of preorder is root
            root = TreeNode(preorder[0])
            partition = inorder.index(preorder[0])
            # the left subtree
            # all inorder elements before partition will be left subtree
            # all preorder elements from index 1 to index partion (including) will be left subtree
            root.left = self.buildTree(preorder[1:partition+1], inorder[:partition])

            # the right subtree
            # all inorder elements after the partition will be the right subtree
            # all preorder elements from partition+1 to the end will be left subtree
            root.right = self.buildTree(preorder[partition+1:], inorder[partition+1:])
            return root
                
        root = buildTree(preorder_list, inorder_list)
        return root
            
            


        

### Alternate Aproach


In serialization, everytime the node is null, we append x
rest times we append the node value

In [21]:
class TreeNode(object):
    def __init__(self, x):
        self.val = x
        self.left = None
        self.right = None

In [22]:
def serialize(root):

    # create an empty array for result
    result = []

    # create a dfs function
    def dfs(root):
        if root is None:
            result.append('x')
            return None

        # if root is not none
        result.append(root.val)
        # call on left subtree 
        # call on right subtree
        dfs(root.left)
        dfs(root.right)

    dfs(root)
    output = ''.join(result)

In [ ]:
def deserialize(output):

    # define dfs
    def dfs(nodes):
        val = next(nodes)
        if val == 'x':
            return None
        root = TreeNode(int(val))
        root.left = dfs(nodes)
        root.right = dfs(nodes)

        return root

    return dfs(iter(output.split()))


### Optimized Approach
while loop with a stack you manage yourself. It's the same idea as the BST validation follow-up. The recursion's call stack overflows, so you keep the pending work in your own stack, which lives on the heap and can hold hundreds of thousands of entries.

serialize is the easy one. The dfs visits a node, then its whole left subtree, then its right subtree. With a stack, you pop a node and record it, then push its children so that the left one gets popped next. That means pushing the right child first and the left child second.

In [23]:
def serialize(self, root):
    stack = [root]
    result = []
    while stack:
        node = stack.pop()
        if node is None:
            result.append('x')
        else:
            result.append(str(node.val))
            stack.append(node.right)   # pushed first, so popped last
            stack.append(node.left)    # pushed second, so popped next
    return ' '.join(result)

The recursion used to remember which node was still waiting for a child. Now you keep that yourself: a stack of (node, which slot is next) pairs.

For each token, the node on top of the stack is the one that's waiting. If it's waiting for its left child, fill the left slot and put the parent back, now waiting for its right. If it's waiting for its right child, fill that slot and the parent is done. Then if the token made a real node, that node goes on top, waiting for its own left child.

Trace it on 1 2 x x 3 4 x x x (the tree with 1 at the root, left child 2, right child 3, and 3 having a left child 4) when you come back, and watch the stack at each step.

Complexity: both are O(n) time. Space is O(n) for the output string or token list, plus the stack, which is O(h) in the worst case.

In [24]:
def deserialize(self, data):
    tokens = data.split()
    if tokens[0] == 'x':
        return None
    root = TreeNode(int(tokens[0]))
    stack = [(root, 'left')]
    for tok in tokens[1:]:
        parent, slot = stack.pop()
        child = None if tok == 'x' else TreeNode(int(tok))
        if slot == 'left':
            parent.left = child
            stack.append((parent, 'right'))   # parent still needs its right child
        else:
            parent.right = child              # parent is now finished
        if child is not None:
            stack.append((child, 'left'))     # new node needs its own children
    return root